In [ ]:
import os

import tarfile


import urllib.request




DOWNLOAD_ROOT = "https://raw.githubusercontent.com/ageron/handson-ml2/master/"


HOUSING_PATH = os.path.join("datasets", "housing")


HOUSING_URL = DOWNLOAD_ROOT + "datasets/housing/housing.tgz"




# function to fetch the data set


def fetch_housing_data(housing_url=HOUSING_URL, housing_path=HOUSING_PATH):


    os.makedirs(housing_path, exist_ok=True)


    tgz_path = os.path.join(housing_path, "housing.tgz")



    urllib.request.urlretrieve(housing_url, tgz_path)


    housing_tgz = tarfile.open(tgz_path)

    housing_tgz.extractall(path=housing_path)
    housing_tgz.close()




# calling function fetch


fetch_housing_data()



In [2]:
import pandas as pd




# function to load the data set into work space


def load_housing_data(housing_path=HOUSING_PATH):


    csv_path = os.path.join(housing_path, "housing.csv")


    return pd.read_csv(csv_path)




# calling function load


housing =load_housing_data()

NameError: name 'HOUSING_PATH' is not defined

In [ ]:
###housing.head()

In [ ]:
housing.info()

In [ ]:
import matplotlib.pyplot as plt


housing.hist(bins=50, figsize=(20,15))

In [ ]:
housing["ocean_proximity"].value_counts()

In [ ]:
housing.describe()

In [ ]:
import numpy as np




def split_train_test(data, test_ratio):


    shuffled_indices = np.random.permutation(len(data))


    test_set_size = int(len(data) * test_ratio)


    test_indices = shuffled_indices[:test_set_size]


    train_indices = shuffled_indices[test_set_size:]


    return data.iloc[train_indices], data.iloc[test_indices]




train_set, test_set = split_train_test(housing, 0.2)


len(train_set)


len(test_set)





In [ ]:
from sklearn.model_selection import train_test_split




train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

In [ ]:
housing["income_cat"] = pd.cut(housing["median_income"],


                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],


                               labels=[1, 2, 3, 4, 5])


housing["income_cat"].hist()





In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit




split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)


for train_index, test_index in split.split(housing, housing["income_cat"]):


    strat_train_set = housing.loc[train_index]


    strat_test_set = housing.loc[test_index]




strat_test_set["income_cat"].value_counts() / len(strat_test_set)



In [ ]:
for set_ in (strat_train_set, strat_test_set):


    set_.drop("income_cat", axis=1, inplace=True)

week 2


In [ ]:
housing = strat_train_set.copy() 




housing.plot(kind="scatter", x="median_income", y="median_house_value", alpha=0.1)



In [ ]:
import matplotlib.pyplot as plt


housing.hist(bins=50, figsize=(20,15))

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.1)

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.4,


             s=housing["population"]/100, label="population", figsize=(10,7),


             c="median_house_value", cmap=plt.get_cmap("jet"), colorbar=True)


plt.legend()

In [ ]:

housing_num = housing.drop('ocean_proximity', axis=1) 


housing_num.head()




corr_matrix = housing_num.corr()
corr_matrix["median_house_value"].sort_values(ascending=False)



In [ ]:
from pandas.plotting import scatter_matrix




attributes = ["median_house_value", "median_income", "total_rooms",


              "housing_median_age"]


scatter_matrix(housing[attributes], figsize=(12, 8))



In [ ]:

housing["rooms_per_household"] = housing["total_rooms"]/housing["households"]


housing["bedrooms_per_room"] = housing["total_bedrooms"]/housing["total_rooms"]


housing["population_per_household"]=housing["population"]/housing["households"]






# since ﷿﷿﷿housing﷿﷿﷿ has a categorial value column ﷿﷿﷿ create a copy for correlation calc


housing_num = housing.drop("ocean_proximity", axis=1)


housing_num.head()


corr_matrix = housing_num.corr()


corr_matrix["median_house_value"].sort_values(ascending=False)





Prepare the Data for Machine Learning Algorithms

Once you've split the data and explored it, the next crucial step is data preparation. This ensures the data is in a form suitable for ML algorithms. The section covers several key steps and techniques. Let﷿﷿﷿s first get a clean copy of our training dataset and labels separately.

In [ ]:

housing = strat_train_set.drop("median_house_value", axis=1)


housing_labels = strat_train_set["median_house_value"].copy()

Handling Missing Values

Most Machine Learning algorithms cannot work with missing features. Some options to take care of them;

Get rid of the corresponding districts.
Get rid of the whole attribute.
Set the values to some value (zero, the mean, the median, etc.).

In [ ]:
housing.dropna(subset=["total_bedrooms"])    # option 1


housing.drop("total_bedrooms", axis=1)       # option 2


median = housing["total_bedrooms"].median()  # option 3


housing["total_bedrooms"].fillna(median, inplace=True)

In [ ]:
from sklearn.impute import SimpleImputer


imputer = SimpleImputer(strategy="median")




housing_num = housing.drop("ocean_proximity", axis=1)


imputer.fit(housing_num)





In [ ]:
X = imputer.transform(housing_num)


housing_tr = pd.DataFrame(X, columns=housing_num.columns,


                          index=housing_num.index)

housing_cat = housing[["ocean_proximity"]]


housing_cat.head(10)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder


ordinal_encoder = OrdinalEncoder()


housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)


housing_cat_encoded[:10]
ordinal_encoder.categories_

In [ ]:
from sklearn.preprocessing import OneHotEncoder




cat_encoder = OneHotEncoder()


housing_cat = housing[["ocean_proximity"]]


housing_cat_1hot = cat_encoder.fit_transform(housing_cat)


housing_cat_1hot.toarray()
cat_encoder.categories_

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin




rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6




class CombinedAttributesAdder(BaseEstimator, TransformerMixin):


    def __init__(self, add_bedrooms_per_room=True): # no *args or **kargs


        self.add_bedrooms_per_room = add_bedrooms_per_room


    def fit(self, X, y=None):


        return self  # nothing else to do


    def transform(self, X):


        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]


        population_per_household = X[:, population_ix] / X[:, households_ix]


        if self.add_bedrooms_per_room:


            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]


            return np.c_[X, rooms_per_household, population_per_household,


                         bedrooms_per_room]




        else:


            return np.c_[X, rooms_per_household, population_per_household]




attr_adder = CombinedAttributesAdder(add_bedrooms_per_room=False)


housing_extra_attribs = attr_adder.transform(housing.values)



In [ ]:
from sklearn.preprocessing import MinMaxScaler


mm_scaler = MinMaxScaler ()




housing_minmax_scaled=MinMaxScaler(housing)

In [ ]:
from sklearn.preprocessing import StandardScaler




s_scaler = StandardScaler()


housing_scaled = s_scaler.fit_transform(housing_num)

In [ ]:
from sklearn.pipeline import Pipeline


from sklearn.preprocessing import StandardScaler




num_pipeline = Pipeline([


        ('imputer', SimpleImputer(strategy="median")),


        ('attribs_adder', CombinedAttributesAdder()),


        ('std_scaler', StandardScaler()),


    ])




housing_num_tr = num_pipeline.fit_transform(housing_num)



In [ ]:

from sklearn.compose import ColumnTransformer




num_attribs = list(housing_num)


cat_attribs = ["ocean_proximity"]




full_pipeline = ColumnTransformer([


        ("num", num_pipeline, num_attribs),


        ("cat", OneHotEncoder(), cat_attribs),


    ])




housing_prepared = full_pipeline.fit_transform(housing)




housing_num_tr = num_pipeline.fit_transform(housing_num)

In [ ]:
from sklearn.linear_model import LinearRegression




lin_reg = LinearRegression()


lin_reg.fit(housing_prepared, housing_labels)


housing_num_tr = num_pipeline.fit_transform(housing_num)

Week 4

In [4]:
from sklearn.datasets import fetch_openml


mnist = fetch_openml('mnist_784', version=1)


X, y = mnist["data"], mnist["target"]


import numpy as np


y = y.astype(np.uint8)


X_train, X_test = X[:60000], X[60000:]


y_train, y_test = y[:60000], y[60000:]


y_train_5 = (y_train == 5)  # True for all 5s, False for all other digits


y_test_5 = (y_test == 5)


from sklearn.linear_model import SGDClassifier


sgd_clf = SGDClassifier(random_state=42)


sgd_clf.fit(X_train, y_train_5)

,"loss loss: {'hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='hinge'The loss function to be used.- 'hinge' gives a linear SVM.- 'log_loss' gives logistic regression, a probabilistic classifier.- 'modified_huber' is another smooth loss that brings tolerance to outliers as well as probability estimates.- 'squared_hinge' is like hinge but is quadratically penalized.- 'perceptron' is the linear loss used by the perceptron algorithm.- The other losses, 'squared_error', 'huber', 'epsilon_insensitive' and 'squared_epsilon_insensitive' are designed for regression but can be useful in classification as well; see :class:`~sklearn.linear_model.SGDRegressor` for a description.More details about the losses formulas can be found in the :ref:`User Guide` and you can find a visualisation of the lossfunctions in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_loss_functions.py`.",'hinge'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True
,"max_iter max_iter: int, default=1000The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the ``fit`` method, and not the:meth:`partial_fit` method.Values must be in the range `[1, inf)`... versionadded:: 0.19",1000
,"tol tol: float or None, default=1e-3The stopping criterion. If it is not None, training will stopwhen (loss > best_loss - tol) for ``n_iter_no_change`` consecutiveepochs.Convergence is checked against the training loss or thevalidation loss depending on the `early_stopping` parameter.Values must be in the range `[0.0, inf)`... versionadded:: 0.19",0.001
,"shuffle shuffle: bool, default=TrueWhether or not the training data should be shuffled after each epoch.",True
,"verbose verbose: int, default=0The verbosity level.Values must be in the range `[0, inf)`.",0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-insensitive loss functions; only if `loss` is'huber', 'epsilon_insensitive', or 'squared_epsilon_insensitive'.For 'huber', determines the threshold at which it becomes lessimportant to get the prediction exactly right.For epsilon-insensitive, any differences between the current predictionand the correct label are ignored if they are less than this threshold.Values must be in the range `[0.0, inf)`.",0.1
,"n_jobs n_jobs: int, default=NoneThe number of CPUs to use to do the OVA (One Versus All, formulti-class problems) computation.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [5]:
from sklearn.model_selection import StratifiedKFold


from sklearn.base import clone




skfolds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)




for train_index, test_index in skfolds.split(X_train, y_train_5):


    clone_clf = clone(sgd_clf)


    X_train_folds = X_train.iloc[train_index]


    y_train_folds = y_train_5.iloc[train_index]


    X_test_fold = X_train.iloc[test_index]


    y_test_fold = y_train_5.iloc[test_index]




    clone_clf.fit(X_train_folds, y_train_folds)


    y_pred = clone_clf.predict(X_test_fold)


    n_correct = sum(y_pred == y_test_fold)


    print(n_correct / len(y_pred))





0.9042
0.9477
0.96785


In [6]:
from sklearn.model_selection import cross_val_score


cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring="accuracy")

array([0.95035, 0.96035, 0.9604 ])

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

y_train_pred = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3)
cm = confusion_matrix(y_train_5, y_train_pred)
print(cm)

tn, fp, fn, tp = cm.ravel()
print('TN:', tn, 'FP:', fp, 'FN:', fn, 'TP:', tp)

In [ ]:
from sklearn.metrics import confusion_matrix


confusion_matrix(y_train_5, y_train_pred)

In [ ]:
y_train_perfect_predictions = y_train_5  # pretend we reached perfection


confusion_matrix(y_train_5, y_train_perfect_predictions)

In [ ]:
from sklearn.metrics import precision_score, recall_score


precision_score(y_train_5, y_train_pred) 


recall_score(y_train_5, y_train_pred) 



In [ ]:
from sklearn.metrics import f1_score


f1_score(y_train_5, y_train_pred)

In [ ]:
# Recalling data visualization from last week's lesson


import matplotlib as mpl


import matplotlib.pyplot as plt


image_number = 0


some_digit = X.iloc[image_number]


some_digit_image = some_digit.values.reshape(28, 28)




plt.imshow(some_digit_image, cmap="binary")


plt.axis("off")


plt.show()


y.iloc[image_number]  # check the label


y_scores = sgd_clf.decision_function([some_digit]) # check the decision score


y_scores

In [ ]:
threshold = 8000


y_some_digit_pred = (y_scores > threshold)


y_some_digit_pred



In [ ]:
y_scores = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3,


                             method="decision_function")


from sklearn.metrics import precision_recall_curve


precisions, recalls, thresholds = precision_recall_curve(y_train_5, y_scores)


def plot_precision_recall_vs_threshold(precisions, recalls, thresholds):


    plt.plot(thresholds, precisions[:-1], "b--", label="Precision")


    plt.plot(thresholds, recalls[:-1], "g-", label="Recall")


    [...] 




plot_precision_recall_vs_threshold(precisions, recalls, thresholds)


plt.show()



In [ ]:
plt.plot(recalls, precisions, "b-", linewidth=2)


plt.xlabel("Recall")


plt.ylabel("Precision")


plt.axis([0, 1, 0, 1])


plt.grid(True)


plt.show()



In [ ]:
threshold_90_precision = thresholds[np.argmax(precisions >= 0.90)] 


y_train_pred_90 = (y_scores >= threshold_90_precision)



In [ ]:
from sklearn.metrics import roc_curve




fpr, tpr, thresholds = roc_curve(y_train_5, y_scores)


def plot_roc_curve(fpr, tpr, label=None):


    plt.plot(fpr, tpr, linewidth=2, label=label)


    plt.plot([0, 1], [0, 1], 'k--') # Dashed diagonal




plot_roc_curve(fpr, tpr)


plt.show()





In [ ]:
from sklearn.metrics import roc_auc_score


roc_auc_score(y_train_5, y_scores)



week 4 part 2


In [ ]:
some_data = housing.iloc[:5]


some_labels = housing_labels.iloc[:5]


some_data_prepared = full_pipeline.transform(some_data)


print("Predictions:", lin_reg.predict(some_data_prepared))


print("Labels:", list(some_labels))



In [ ]:
# Week 4


from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


housing_predictions = lin_reg.predict(housing_prepared)




lin_mse = mean_squared_error(housing_labels, housing_predictions)


lin_rmse = np.sqrt(lin_mse)


lin_mae = mean_absolute_error(housing_labels, housing_predictions)


lin_r2 = r2_score(housing_labels, housing_predictions)




print("RMSE:", lin_rmse)


print("MAE:", lin_mae)


print("R﷿﷿:", lin_r2)







In [ ]:
from sklearn.model_selection import cross_val_score


scores = cross_val_score(lin_reg, housing_prepared, housing_labels,


                         scoring="neg_mean_squared_error", cv=10)


lin_rmse_scores = np.sqrt(-scores)


lin_rmse_scores

In [ ]:
def display_scores(scores):


    print("Scores:", scores)


    print("Mean:", scores.mean())


    print("Standard deviation:", scores.std())


...


display_scores(lin_rmse_scores)



In [ ]:
X_test = strat_test_set.drop("median_house_value", axis=1)


y_test = strat_test_set["median_house_value"].copy()




X_test_prepared = full_pipeline.transform(X_test)


final_predictions = lin_reg.predict(X_test_prepared)




final_mse = mean_squared_error(y_test, final_predictions)


final_rmse = np.sqrt(final_mse)




print("Final RMSE:", final_rmse)